In [1]:
%cd ../.
import os, sys

/home/gtamo/CDD_Vault_API


In [2]:
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.insert(0, os.path.abspath('python'))   # cell [0] cd'd to repo root

import pandas as pd
import re
from pathlib import Path
from IPython.display import display, Markdown

# custom function
from convert_dataset import convert_to_target_format,normalize_ori_columns

In [3]:
ori = pd.read_csv('data/MDR1_ori.csv')

In [4]:
col_ori = pd.read_excel('data/mdr1_conversion.xlsx', sheet_name='original') # sheet ori
col_target = pd.read_excel('data/mdr1_conversion.xlsx', sheet_name='target') # sheet target

## Long → wide pivot (no-inhibitor + with-inhibitor side-by-side)

Pipeline:

1. **Strip the `MDR1-MDCK II: ` CDD prefix** from `ori`'s column names — it just records the protocol source and isn't part of the target schema.
2. **Keep only `id_cols + col_ori['Name']`** — drops upstream metadata like `Run Date`, `Run Lab`, etc.
3. **Pivot** using `col_target` as the canonical wide layout: each `col_ori` base column becomes either `<base>` (no-inhibitor row) or `<base> - PgP inhibitor` (with-inhibitor row).

You supply `inhibitor_mask` — a boolean Series aligned to `ori.index` that's `True` for rows measured **with** PgP inhibitor. The mask can come from any column in the original `ori` (even ones not in `col_ori`, since the mask is computed before filtering).

In [5]:
# Canonical home: python/convert_dataset.py — edits there propagate via %autoreload 2.
# Unit tests for this function live in tests/test_convert_dataset.py.


### Example usage

Build the mask from whichever column actually encodes the condition. Two common shapes:

In [6]:
# Build the boolean mask from whichever column in the ORIGINAL `ori`
# (still with the 'MDR1-MDCK II: ' prefix) encodes the inhibitor condition.
# Two common shapes:

# Option A — explicit boolean Series from a known column
inhibitor_mask = ori['MDR1-MDCK II: Cell line'].astype(str).str.contains(r'inhi[bn]itor', case=False, na=False, regex=True)

# Option B — derive from the comment / study-number field
# inhibitor_mask = ori['MDR1-MDCK II: Study number'].astype(str).str.contains('PgP', na=False)

target = convert_to_target_format(ori, col_ori, col_target, inhibitor_mask)
target.shape, list(target.columns)

WARN: duplicate id_cols in no-inhibitor rows — keeping first
WARN: duplicate id_cols in with-inhibitor rows — keeping first


((90, 28),
 ['Molecule Name',
  'Batch Molecule-Batch ID',
  'Study number',
  'Study number - PgP inhibitor',
  'Date',
  'Date - PgP inhibitor',
  'Cell line',
  'Cell line - PgP inhibitor',
  'Mean Papp A to B',
  'Mean Papp A to B - PgP Inhibitor',
  'Mean Papp B to A',
  'Mean Papp B to A - PgP inhibitor',
  'Efflux ratio',
  'Efflux ratio - PgP inhibitor',
  'Mean %Solution Recovery A to B',
  'Mean %Solution Recovery A to B - PgP inhibitor',
  'Mean %Solution Recovery B to A',
  'Mean %Solution Recovery B to A - PgP inhibitor',
  'Provider Comment',
  'Provider Comment - PgP inhibitor',
  'Serac Comment',
  'Serac Comment - PgP inhibitor',
  'Permeability Class',
  'Permeability Class - PgP inhibitor',
  'Pgp substrate',
  'Pgp substrate - Pgp inhibitor',
  'Provider Name',
  'Provider Name - PgP inhibitor'])

## Visualize the conversion

Eyeball what `convert_to_target_format` did: render the *before* (rows of `ori` for one compound) and the *after* (one row of `target` for the same compound), with color coding:

- **blue** background → no-inhibitor condition (in `ori`) / base columns (in `target`)
- **red** background → with-inhibitor condition (in `ori`) / `- PgP inhibitor` columns (in `target`)
- **gray** background → identifier columns (`Molecule Name`, `Batch Molecule-Batch ID`)

The function is generic — point it at your real `ori` / `target` later by passing them as the first two arguments.

> **Privacy note:** rendered output contains the actual values for the chosen compound. If you commit the notebook, clear outputs first (`jupyter nbconvert --clear-output --inplace …`) to keep chemistry local.

In [7]:
# visualize_conversion lives in python/convert_dataset.py (with %autoreload
# 2, edits there propagate). The trailing helper _render_one is private and
# isn't imported here.
from convert_dataset import visualize_conversion

In [8]:
# Real-data flow: build the mask, run the conversion, visualize the first 2 compounds.
# NOTE: if every inhibitor cell in target_df comes out NaN, your mask is matching
# zero rows — see chat reply for the debug snippet to find the right column.

# Option A — explicit boolean Series from a known column
inhibitor_mask = ori['MDR1-MDCK II: Cell line'].astype(str).str.contains(r'inhi[bn]itor', case=False, na=False, regex=True)
target_df = convert_to_target_format(ori, col_ori, col_target, inhibitor_mask)

# check whether all col_targets
# Sanity check: target_df has id_cols + col_target['Name'] in order.
assert list(target_df.columns) == ['Molecule Name', 'Batch Molecule-Batch ID'] + list(col_target['Name'])
target_df.head(3)

# Pass a list to render multiple compounds back-to-back.
first_two = target_df['Molecule Name'].iloc[:1].tolist() + ['SRB-0005908']
visualize_conversion(
    ori, target_df, first_two,
    col_ori, col_target, inhibitor_mask)

WARN: duplicate id_cols in no-inhibitor rows — keeping first
WARN: duplicate id_cols in with-inhibitor rows — keeping first


### Conversion view for `SRB-0002248`

**Before** (`ori`, one row per condition):

,Molecule Name,Batch Molecule-Batch ID,Study number,Date,Cell line,Mean Papp A to B,Mean Papp B to A,Efflux ratio,Mean %Solution Recovery A to B,Mean %Solution Recovery B to A,Provider Comment,Serac Comment,Permeability Class,Pgp substrate,Provider Name
178,SRB-0002248,SRB-0002248-001,426354-20251024-MDR1,nan,MDR1-MDCKⅡ,1.57,10.60,6.79,85.70,93.40,---,nan,Moderate,nan,wuxi
179,SRB-0002248,SRB-0002248-001,426354-20251024-MDR1,nan,MDR1-MDCKⅡ+ PgP inhinitor,2.85,3.70,1.30,98.20,101.00,---,nan,---,nan,wuxi


**After** (`target`, one row per compound):

...

### Conversion view for `SRB-0005908`

**Before** (`ori`, one row per condition):

,Molecule Name,Batch Molecule-Batch ID,Study number,Date,Cell line,Mean Papp A to B,Mean Papp B to A,Efflux ratio,Mean %Solution Recovery A to B,Mean %Solution Recovery B to A,Provider Comment,Serac Comment,Permeability Class,Pgp substrate,Provider Name
140,SRB-0005908,SRB-0005908-001,426354-2025121601-MDR1,nan,MDR1-MDCKⅡ,< 0.38,3.90,> 10.10,< 67.00,61.50,---,nan,Low,nan,wuxi
141,SRB-0005908,SRB-0005908-001,426354-2025121601-MDR1,nan,MDR1-MDCKⅡ+ PgP inhinitor,< 0.38,1.20,> 3.16,< 68.40,57.60,---,nan,---,nan,wuxi


**After** (`target`, one row per compound):

,Molecule Name,Batch Molecule-Batch ID,Study number,Study number - PgP inhibitor,Date,Date - PgP inhibitor,Cell line,Cell line - PgP inhibitor,Mean Papp A to B,Mean Papp A to B - PgP Inhibitor,Mean Papp B to A,Mean Papp B to A - PgP inhibitor,Efflux ratio,Efflux ratio - PgP inhibitor,Mean %Solution Recovery A to B,Mean %Solution Recovery A to B - PgP inhibitor,Mean %Solution Recovery B to A,Mean %Solution Recovery B to A - PgP inhibitor,Provider Comment,Provider Comment - PgP inhibitor,Serac Comment,Serac Comment - PgP inhibitor,Permeability Class,Permeability Class - PgP inhibitor,Pgp substrate,Pgp substrate - Pgp inhibitor,Provider Name,Provider Name - PgP inhibitor
20,SRB-0005908,SRB-0005908-001,426354-2025121601-MDR1,426354-2025121601-MDR1,nan,nan,MDR1-MDCKⅡ,MDR1-MDCKⅡ+ PgP inhinitor,< 0.38,< 0.38,3.900000,1.200000,> 10.10,> 3.16,< 67.00,< 68.40,61.500000,57.600000,---,---,nan,nan,Low,---,nan,nan,wuxi,wuxi


In [9]:
# Check that target_df has exactly the expected columns (id_cols + col_target['Name']).
id_cols = ['Molecule Name', 'Batch Molecule-Batch ID']
expected_cols = id_cols + list(col_target['Name'])
all_match = list(target_df.columns) == expected_cols

# If False, diff so you can see what's off (order or membership).
if not all_match:
    missing = [c for c in expected_cols if c not in target_df.columns]
    extra   = [c for c in target_df.columns if c not in expected_cols]
    print(f'missing_from_target_df ({len(missing)}): {missing}')
    print(f'extra_in_target_df    ({len(extra)}): {extra}')
all_match

True

In [10]:
target_df.to_csv('output/pivoted_mdr1.csv',index=False,sep=',')

## Round-trip test: pivoted CSV ↔ original `ori`

Saves `target_df` to `output/pivoted_mdr1.csv`, reverse-pivots it back to long form, and compares cell-by-cell against the original `ori` (normalized to the same shape).

If the forward pivot ever silently drops or alters data, this test catches it. Output is metadata only — counts, column names, percentages — no compound values.

In [11]:
# test_pivot_round_trip also lives in python/convert_dataset.py — see
# next cell for the invocation on real data.
from convert_dataset import test_pivot_round_trip

In [12]:
# Run on the real data — assumes ori, target_df, col_ori, col_target, and
# inhibitor_mask are already in scope from cells above.
test_pivot_round_trip(target_df, ori, col_ori, col_target, inhibitor_mask)

saved: output/pivoted_mdr1.csv  shape=(90, 28)
note: 2 duplicate (id, condition) row(s) in ori were deduped to match the forward pivot. Inspect locally with: ori.assign(_cond=inhibitor_mask).loc[lambda d: d.duplicated(subset=['Molecule Name', 'Batch Molecule-Batch ID'] + ['_cond'], keep=False)]
ori rows (normalized + filtered): 178
recovered rows (reverse-pivot):    178
columns compared:  16
total cells:       2848
differing cells:   0
match rate:        100.0000%

PASS: pivoted_mdr1.csv reverses cleanly into the original ori data.


True

### 2026-06-08 New data upload from Wuxi

In [42]:
from pathlib import Path

NEW_XLSX = 'data/426354-2026052701-MDR1_report_20260603.xlsx'
NEW_SHEET = 'Upload'
NEW_OUT = 'output/20260608_pivoted_mdr1_fixed.csv'

# Rename map: the new export uses bare column names (no MDR1-MDCK II: prefix,
# no unit suffix) and has minor casing / whitespace differences from col_ori.
# Bring everything to the canonical names col_ori['Name'] uses so the
# conversion pipeline finds its columns. Pass ori_prefix='' below since
# the renamed columns no longer carry a prefix to strip.
NEW_COLUMN_RENAME = {
    # id columns
    'Compound':                        'Molecule Name',
    'batch':                           'Batch Molecule-Batch ID',
    # col_ori names
    'study  number':                   'Study number',          # double space
    'Cell Line':                       'Cell line',             # capital L
    'Mean  Papp A to B':               'Mean Papp A to B',      # double space
    'Mean  Papp B to A':               'Mean Papp B to A',      # double space
    'Efflux Ratio':                    'Efflux ratio',          # capital R
    'Mean % Solution Recovery A to B': 'Mean %Solution Recovery A to B',  # extra space
    'Mean % Solution Recovery B to A': 'Mean %Solution Recovery B to A',  # extra space
    'Provider name':                   'Provider Name',         # lowercase n
    'Permeability':                    'Permeability Class',    # shorter in new
    'Pgp substrate':                   'Pgp substrate',         # exact match (no-op)
    'comments':                        'Provider Comment',      # best guess — single comment col in new file
    # Missing in new file (col_ori expects them; they'll come out NaN):
    #   - 'Date'
    #   - 'Serac Comment'
}

# Use the openpyxl-backed reader so that cells whose Excel display format
# is e.g. '\< 0.000' (i.e. value is just 0.384 but rendered as '< 0.384')
# come through as the qualifier text rather than the raw float.
from convert_dataset import read_xlsx_preserving_qualifiers
new_ori_raw = read_xlsx_preserving_qualifiers(NEW_XLSX, NEW_SHEET)
new_ori = new_ori_raw.rename(columns=NEW_COLUMN_RENAME)

# 'batch' was renamed to 'Batch Molecule-Batch ID' above, but its values are
# just the integer batch number (1, 2, ...). The canonical id is
# '<Molecule Name>-<NNN>' where NNN is the zero-padded batch number
# (e.g. 'SRB-XXXXXXX-001'). Build that explicitly here.
_batch_num = new_ori['Batch Molecule-Batch ID']
if _batch_num.isna().any():
    print(f"WARN: {int(_batch_num.isna().sum())} batch number(s) are NaN — "
          f"defaulting them to 1")
new_ori['Batch Molecule-Batch ID'] = (
    new_ori['Molecule Name'].astype(str)
    + '-'
    + _batch_num.fillna(1).astype(int).astype(str).str.zfill(3)
)

print(f'loaded: {NEW_XLSX}  sheet={NEW_SHEET!r}  shape={new_ori.shape}')
print(f'columns after rename: {list(new_ori.columns)}')

# Build the mask from the renamed column. Same logic as the original ori path.
new_mask = (
    new_ori['Cell line']
        .astype(str)
        .str.contains(r'inhi[bn]itor', case=False, na=False, regex=True)
    )
print(f'inhibitor_mask: True={int(new_mask.sum())} / {len(new_mask)}')

# Convert. ori_prefix='' because the columns are already bare after the rename.
new_target = convert_to_target_format(
    new_ori, col_ori, col_target, new_mask,
    ori_prefix='',
)

Path(NEW_OUT).parent.mkdir(parents=True, exist_ok=True)
new_target.to_csv(NEW_OUT, index=False)
print(f'\nsaved: {NEW_OUT}  shape={new_target.shape}')

# Optional round-trip check on the new export:
# test_pivot_round_trip(new_target, new_ori, col_ori, col_target, new_mask,
#                       ori_prefix='', output_path=NEW_OUT)

/home/gtamo/CDD_Vault_API/cdd/lib/python3.12/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


loaded: data/426354-2026052701-MDR1_report_20260603.xlsx  sheet='Upload'  shape=(38, 13)
columns after rename: ['Molecule Name', 'Batch Molecule-Batch ID', 'Study number', 'Cell line', 'Mean Papp A to B', 'Mean Papp B to A', 'Efflux ratio', 'Mean %Solution Recovery A to B', 'Mean %Solution Recovery B to A', 'Provider Name', 'Permeability Class', 'Pgp substrate', 'Provider Comment']
inhibitor_mask: True=19 / 38
WARN: 2 col_ori name(s) not found in ori (after prefix strip): ['Date', 'Serac Comment']

saved: output/20260608_pivoted_mdr1_fixed.csv  shape=(19, 28)


In [43]:
# Round-trip the new pivot back to long form and verify lossless against new_ori.
# ori_prefix='' because the new file's columns are already bare (no 'MDR1-MDCK II: ').
test_pivot_round_trip(
    new_target, new_ori, col_ori, col_target, new_mask,
    ori_prefix='',
    output_path='output/20260608_pivoted_mdr1.csv',
)

saved: output/20260608_pivoted_mdr1.csv  shape=(19, 28)
ori rows (normalized + filtered): 38
recovered rows (reverse-pivot):    38
columns compared:  14
total cells:       532
differing cells:   0
match rate:        100.0000%

PASS: pivoted_mdr1.csv reverses cleanly into the original ori data.


True

In [44]:
# Final sanity check: the two pivoted CSVs share the exact same column layout
# (same names, same order). Lets downstream code pd.concat them safely.
from convert_dataset import check_columns_match

old_cols = pd.read_csv('output/pivoted_mdr1.csv', nrows=0).columns.tolist()
new_cols = pd.read_csv('output/20260608_pivoted_mdr1.csv', nrows=0).columns.tolist()

assert check_columns_match(old_cols, new_cols), (
    'Column names / order do not match between the two pivoted CSVs.'
)

old columns: 28   new columns: 28   identical_and_in_order: True


In [ ]:
# Sanity-check the pivoted output: id format, base/inhibitor consistency,
# efflux-ratio = Papp B→A / Papp A→B, uniqueness. See convert_dataset.py
# for the full check list and tunable thresholds.
from convert_dataset import validate_pivoted_output

validate_pivoted_output(new_target)